# Assignment 8 — Pre-trained BERT for Sentiment Analysis

**Platform:** Google Colab &nbsp;|&nbsp; **Suggested runtime:** GPU  
**How to use:** Run the cells from top to bottom. Change the small experiment
constants when more training time is available.

This workbook is written as a compact college assignment: it explains the
problem, implements the method, evaluates the result, and records the main
observations.


## Problem and method

Predict whether an English sentence expresses positive or negative
sentiment. We load a BERT-base model already fine-tuned on SST-2 and
evaluate it on a reproducible sample from the SST-2 validation set.

BERT tokenizes text into subword units, adds special tokens, processes
context bidirectionally with self-attention, and sends the pooled
representation to a classification head.


In [ ]:
%pip install -q -U transformers datasets accelerate
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch

from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

MODEL_NAME = "textattack/bert-base-uncased-SST-2"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME).to(device)
model.eval()
print("Device:", device)


In [ ]:
dataset = load_dataset("glue", "sst2", split="validation")
sample = dataset.shuffle(seed=SEED).select(range(min(500, len(dataset))))
display(pd.DataFrame(sample[:5]))

@torch.no_grad()
def predict_texts(texts, batch_size=32):
    predictions, positive_probabilities = [], []
    for start in range(0, len(texts), batch_size):
        batch = texts[start:start + batch_size]
        encoded = tokenizer(
            batch, padding=True, truncation=True, max_length=128,
            return_tensors="pt"
        ).to(device)
        probabilities = model(**encoded).logits.softmax(dim=-1)
        predictions.extend(probabilities.argmax(dim=-1).cpu().tolist())
        positive_probabilities.extend(probabilities[:, 1].cpu().tolist())
    return np.array(predictions), np.array(positive_probabilities)

texts = sample["sentence"]
y_true = np.array(sample["label"])
y_pred, positive_probability = predict_texts(texts)


In [ ]:
accuracy = accuracy_score(y_true, y_pred)
precision, recall, f1, _ = precision_recall_fscore_support(
    y_true, y_pred, average="binary", zero_division=0
)
print(pd.Series({"accuracy": accuracy, "precision": precision,
                 "recall": recall, "f1": f1}).round(3))

cm = confusion_matrix(y_true, y_pred)
sns.heatmap(cm, annot=True, fmt="d", cmap="Purples",
            xticklabels=["negative", "positive"],
            yticklabels=["negative", "positive"])
plt.xlabel("Predicted"); plt.ylabel("Actual"); plt.title("BERT confusion matrix")
plt.show()


In [ ]:
review = pd.DataFrame({
    "text": texts, "actual": y_true, "predicted": y_pred,
    "positive_probability": positive_probability,
})
mistakes = review[review.actual != review.predicted].copy()
mistakes["confidence"] = np.where(
    mistakes.predicted == 1,
    mistakes.positive_probability,
    1 - mistakes.positive_probability,
)
display(mistakes.sort_values("confidence", ascending=False).head(10))

custom_text = ["The story was slow, but the ending was wonderful."]
label, probability = predict_texts(custom_text)
print(custom_text[0], "->", ["negative", "positive"][label[0]],
      f"(positive probability={probability[0]:.3f})")


## Discussion

F1 balances precision and recall. Inspecting confident mistakes is useful
because sentiment can depend on sarcasm, negation, mixed opinions, or
missing context. This is transfer learning: BERT learned general language
representations first and was later adapted to sentiment labels.


## Conclusion

The experiment above provides a complete training and evaluation workflow. The
printed metrics and plots are the result for the current run and should be used
to identify the strongest behaviour, the main limitation, and one justified
improvement. Exact values may vary slightly because neural-network training is
stochastic.
